# PCL Detection: Enhanced DeBERTa-v3 with Focal Loss

**SemEval 2022 Task 4 Subtask 1 — Binary Classification of Patronizing and Condescending Language**

This notebook implements our proposed approach:
1. **Community-aware input representation** — prepend `[Community: keyword]` to each text
2. **Focal loss with label smoothing** — handles severe class imbalance (~9.5% positive) without discarding data
3. **Layerwise learning rate decay** — lower transformer layers learn slower, preserving general knowledge
4. **Threshold optimization** — systematic F1-maximizing threshold search

Model: `microsoft/deberta-v3-large`

**Note:** The full multi-seed ensemble is run via `src/train.py`. This notebook provides a single-seed reference implementation.

**To run on Colab**: Runtime → Change runtime type → GPU (T4)

## 0. Setup & Installation

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install transformers datasets accelerate scikit-learn -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be slow)")

print(f"PyTorch version: {torch.__version__}")

## 1. Configuration

In [ ]:
# === Model Configuration ===
MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LENGTH = 256
COMMUNITY_AWARE = True  # Prepend [Community: keyword] to input

# === Training Configuration ===
NUM_EPOCHS = 5
BATCH_SIZE = 4           # Smaller for large model
GRADIENT_ACCUMULATION = 8  # Effective batch size = 32
LEARNING_RATE = 1e-5     # Smaller for large model
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2

# === Focal Loss Configuration ===
FOCAL_ALPHA = 0.75       # Weight for positive class
FOCAL_GAMMA = 2.0        # Focusing parameter
LABEL_SMOOTHING = 0.1    # Soft targets for better calibration

# === Layerwise LR Decay ===
LAYERWISE_LR_DECAY = 0.95  # Lower layers get smaller LR

# === Paths ===
# Adjust these for your environment (Colab vs local)
DATA_DIR = "../data"  # Relative to BestModel/
TSV_PATH = os.path.join(DATA_DIR, "dontpatronizeme_pcl.tsv")
TRAIN_SPLIT_PATH = os.path.join(DATA_DIR, "practice-splits", "train_semeval_parids-labels.csv")
DEV_SPLIT_PATH = os.path.join(DATA_DIR, "practice-splits", "dev_semeval_parids-labels.csv")
TEST_PATH = os.path.join(DATA_DIR, "test", "task4_test.tsv")
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Data Loading & Preprocessing

In [ ]:
def load_pcl_dataset(tsv_path):
    """Load full dataset, skipping 4-line disclaimer."""
    df = pd.read_csv(
        tsv_path, sep="\t", skiprows=4, header=None,
        names=["par_id", "art_id", "keyword", "country", "text", "label"],
    )
    df["par_id"] = df["par_id"].astype(str)
    df["binary_label"] = (df["label"] >= 2).astype(int)
    return df


def format_input(text, keyword, community_aware=True):
    """Optionally prepend community context."""
    if community_aware:
        return f"[Community: {keyword}] {text}"
    return text


# Load data
full_df = load_pcl_dataset(TSV_PATH)
train_ids = pd.read_csv(TRAIN_SPLIT_PATH)
dev_ids = pd.read_csv(DEV_SPLIT_PATH)

train_ids["par_id"] = train_ids["par_id"].astype(str)
dev_ids["par_id"] = dev_ids["par_id"].astype(str)

# Build train and dev dataframes
train_df = full_df[full_df["par_id"].isin(train_ids["par_id"])].copy()
dev_df = dev_ids[["par_id"]].merge(full_df, on="par_id", how="left").copy()

# Apply community-aware formatting
train_df["input_text"] = train_df.apply(
    lambda r: format_input(r["text"], r["keyword"], COMMUNITY_AWARE), axis=1
)
dev_df["input_text"] = dev_df.apply(
    lambda r: format_input(r["text"], r["keyword"], COMMUNITY_AWARE), axis=1
)

# Load test set
test_df = pd.read_csv(
    TEST_PATH, sep="\t", header=None,
    names=["par_id", "art_id", "keyword", "country", "text"],
)
test_df["par_id"] = test_df["par_id"].astype(str)
test_df["input_text"] = test_df.apply(
    lambda r: format_input(r["text"], r["keyword"], COMMUNITY_AWARE), axis=1
)

print(f"Train: {len(train_df)} samples ({train_df['binary_label'].sum()} positive, {train_df['binary_label'].mean():.1%})")
print(f"Dev:   {len(dev_df)} samples ({dev_df['binary_label'].sum()} positive, {dev_df['binary_label'].mean():.1%})")
print(f"Test:  {len(test_df)} samples (labels hidden)")

In [ ]:
# Create internal train/validation split for early stopping
train_split, val_split = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df["binary_label"]
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print(f"Internal train: {len(train_split)} ({train_split['binary_label'].sum()} positive)")
print(f"Internal val:   {len(val_split)} ({val_split['binary_label'].sum()} positive)")

## 3. Tokenization & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PCLDataset(Dataset):
    """Dataset for PCL classification."""

    def __init__(self, texts, labels=None, tokenizer=None, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# Create datasets
train_dataset = PCLDataset(
    train_split["input_text"].tolist(),
    train_split["binary_label"].tolist(),
    tokenizer, MAX_LENGTH,
)
val_dataset = PCLDataset(
    val_split["input_text"].tolist(),
    val_split["binary_label"].tolist(),
    tokenizer, MAX_LENGTH,
)
dev_dataset = PCLDataset(
    dev_df["input_text"].tolist(),
    dev_df["binary_label"].tolist(),
    tokenizer, MAX_LENGTH,
)
test_dataset = PCLDataset(
    test_df["input_text"].tolist(),
    labels=None,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print(f"Datasets created. Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Sample encoding length: {train_dataset[0]['input_ids'].shape}")

## 4. Focal Loss & Custom Trainer

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss with label smoothing for binary classification with class imbalance.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    - alpha: weighting factor for the positive class (default 0.75)
    - gamma: focusing parameter that reduces loss for well-classified examples (default 2.0)
    - label_smoothing: soft targets for better calibration (default 0.1)
    """

    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits, targets, reduction="none",
            label_smoothing=self.label_smoothing,
        )
        p_t = torch.exp(-ce_loss)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_weight = (1 - p_t) ** self.gamma
        return (alpha_t * focal_weight * ce_loss).mean()


class ImprovedTrainer(Trainer):
    """HuggingFace Trainer with focal loss + layerwise LR decay."""

    def __init__(self, focal_alpha=0.75, focal_gamma=2.0, label_smoothing=0.0,
                 layerwise_lr_decay=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(
            alpha=focal_alpha, gamma=focal_gamma, label_smoothing=label_smoothing
        )
        self.layerwise_lr_decay = layerwise_lr_decay

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.layerwise_lr_decay is not None and self.layerwise_lr_decay < 1.0:
            return self._create_layerwise_optimizer()
        return super().create_optimizer()

    def _create_layerwise_optimizer(self):
        model = self.model
        decay = self.layerwise_lr_decay
        base_lr = self.args.learning_rate
        no_decay = ["bias", "LayerNorm.weight", "layernorm.weight"]
        num_layers = model.config.num_hidden_layers
        optimizer_grouped_parameters = []
        assigned_params = set()

        # Embedding parameters (lowest LR)
        embed_lr = base_lr * (decay ** num_layers)
        embed_d, embed_nd = [], []
        for n, p in model.named_parameters():
            if "embeddings" in n:
                assigned_params.add(n)
                (embed_nd if any(nd in n for nd in no_decay) else embed_d).append(p)
        if embed_d:
            optimizer_grouped_parameters.append({"params": embed_d, "weight_decay": WEIGHT_DECAY, "lr": embed_lr})
        if embed_nd:
            optimizer_grouped_parameters.append({"params": embed_nd, "weight_decay": 0.0, "lr": embed_lr})

        # Encoder layers (increasing LR)
        for i in range(num_layers):
            layer_lr = base_lr * (decay ** (num_layers - i - 1))
            layer_d, layer_nd = [], []
            for n, p in model.named_parameters():
                if f"encoder.layer.{i}." in n:
                    assigned_params.add(n)
                    (layer_nd if any(nd in n for nd in no_decay) else layer_d).append(p)
            if layer_d:
                optimizer_grouped_parameters.append({"params": layer_d, "weight_decay": WEIGHT_DECAY, "lr": layer_lr})
            if layer_nd:
                optimizer_grouped_parameters.append({"params": layer_nd, "weight_decay": 0.0, "lr": layer_lr})

        # Head (highest LR)
        head_d, head_nd = [], []
        for n, p in model.named_parameters():
            if n not in assigned_params:
                (head_nd if any(nd in n for nd in no_decay) else head_d).append(p)
        if head_d:
            optimizer_grouped_parameters.append({"params": head_d, "weight_decay": WEIGHT_DECAY, "lr": base_lr})
        if head_nd:
            optimizer_grouped_parameters.append({"params": head_nd, "weight_decay": 0.0, "lr": base_lr})

        self.optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
        print(f"Layerwise LR decay={decay}: embed={embed_lr:.2e}, head={base_lr:.2e}")
        return self.optimizer


def compute_metrics(eval_pred):
    """Compute F1, precision, recall for the positive class."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds, pos_label=1),
        "precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "recall": recall_score(labels, preds, pos_label=1, zero_division=0),
    }


print("Focal loss with label smoothing and layerwise LR decay defined.")

## 5. Model Training

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# Enable gradient checkpointing for large model
if "large" in MODEL_NAME.lower():
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# Create trainer with focal loss + layerwise LR decay
trainer = ImprovedTrainer(
    focal_alpha=FOCAL_ALPHA,
    focal_gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING,
    layerwise_lr_decay=LAYERWISE_LR_DECAY,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f"Training {MODEL_NAME} for up to {NUM_EPOCHS} epochs...")
print(f"Focal loss: alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING}")
print(f"Layerwise LR decay: {LAYERWISE_LR_DECAY}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

In [ ]:
# Train!
train_result = trainer.train()

# Print training summary
print("\n=== Training Complete ===")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
# Evaluate on internal validation set
val_metrics = trainer.evaluate(val_dataset)
print("\n=== Internal Validation Results ===")
print(f"F1:        {val_metrics['eval_f1']:.4f}")
print(f"Precision: {val_metrics['eval_precision']:.4f}")
print(f"Recall:    {val_metrics['eval_recall']:.4f}")

## 6. Threshold Optimization

Instead of using the default threshold of 0.5, we search for the threshold that maximizes F1 on the official dev set.

In [ ]:
def find_optimal_threshold(trainer, dataset, labels):
    """
    Search for the classification threshold that maximizes F1.
    
    Returns:
        best_threshold: float
        best_f1: float
        thresholds: list of tested thresholds
        f1_scores: list of corresponding F1 scores
    """
    # Get prediction probabilities
    outputs = trainer.predict(dataset)
    logits = outputs.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    
    # Search thresholds
    thresholds = np.arange(0.10, 0.90, 0.01)
    f1_scores = []
    
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        f1_scores.append(f1)
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    return best_threshold, best_f1, thresholds, f1_scores, probs


# Optimize threshold on the official dev set
dev_labels = dev_df["binary_label"].values
best_threshold, best_f1, thresholds, f1_scores, dev_probs = find_optimal_threshold(
    trainer, dev_dataset, dev_labels
)

print(f"\n=== Threshold Optimization ===")
print(f"Optimal threshold: {best_threshold:.2f}")
print(f"Best F1 (dev):     {best_f1:.4f}")
print(f"Default F1 (0.5):  {f1_score(dev_labels, (dev_probs >= 0.5).astype(int), pos_label=1):.4f}")
print(f"Baseline F1:       0.4800")

In [ ]:
# Plot threshold vs F1 curve
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, f1_scores, 'b-', linewidth=2)
ax.axvline(x=best_threshold, color='r', linestyle='--', label=f'Optimal: {best_threshold:.2f}')
ax.axvline(x=0.5, color='gray', linestyle=':', label='Default: 0.50')
ax.axhline(y=0.48, color='orange', linestyle=':', alpha=0.7, label='Baseline: 0.48')
ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('F1 Score (Positive Class)', fontsize=12)
ax.set_title('Threshold Optimization for PCL Detection', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join("..", "report", "figures", "threshold_optimization.png"), dpi=150)
plt.show()
print("Saved threshold_optimization.png")

## 7. Full Evaluation on Official Dev Set

In [ ]:
# Generate predictions with optimal threshold
dev_preds = (dev_probs >= best_threshold).astype(int)

print("=== Official Dev Set Results ===")
print(f"Threshold: {best_threshold:.2f}")
print(f"F1:        {f1_score(dev_labels, dev_preds, pos_label=1):.4f}")
print(f"Precision: {precision_score(dev_labels, dev_preds, pos_label=1):.4f}")
print(f"Recall:    {recall_score(dev_labels, dev_preds, pos_label=1):.4f}")
print()
print("Classification Report:")
print(classification_report(dev_labels, dev_preds, target_names=["No PCL", "PCL"]))
print("Confusion Matrix:")
cm = confusion_matrix(dev_labels, dev_preds)
print(f"  TN={cm[0,0]:5d}  FP={cm[0,1]:5d}")
print(f"  FN={cm[1,0]:5d}  TP={cm[1,1]:5d}")

## 8. Generate Prediction Files

In [ ]:
# === dev.txt ===
# Order must match par_ids in dev_semeval_parids-labels.csv
# dev_df is already in that order (we merged on dev_ids ordering)
dev_txt_path = os.path.join("..", "dev.txt")
with open(dev_txt_path, "w") as f:
    for pred in dev_preds:
        f.write(f"{pred}\n")
print(f"Saved {dev_txt_path} ({len(dev_preds)} predictions)")

# === test.txt ===
# Get test predictions
test_outputs = trainer.predict(test_dataset)
test_logits = test_outputs.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1)[:, 1].numpy()
test_preds = (test_probs >= best_threshold).astype(int)

test_txt_path = os.path.join("..", "test.txt")
with open(test_txt_path, "w") as f:
    for pred in test_preds:
        f.write(f"{pred}\n")
print(f"Saved {test_txt_path} ({len(test_preds)} predictions)")

# Verify counts
print(f"\ndev.txt: {sum(dev_preds)} positive / {len(dev_preds)} total")
print(f"test.txt: {sum(test_preds)} positive / {len(test_preds)} total")
assert len(dev_preds) == 2094, f"Expected 2094 dev predictions, got {len(dev_preds)}"
assert len(test_preds) == 3832, f"Expected 3832 test predictions, got {len(test_preds)}"

## 9. Error Analysis

In [ ]:
# Confusion matrix visualization
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: No PCL", "Pred: PCL"],
            yticklabels=["True: No PCL", "True: PCL"], ax=ax)
ax.set_title("Confusion Matrix (Official Dev Set)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join("..", "report", "figures", "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# False Positive and False Negative analysis
dev_analysis = dev_df.copy()
dev_analysis["pred"] = dev_preds
dev_analysis["prob"] = dev_probs
dev_analysis["correct"] = dev_analysis["binary_label"] == dev_analysis["pred"]

false_positives = dev_analysis[(dev_analysis["binary_label"] == 0) & (dev_analysis["pred"] == 1)]
false_negatives = dev_analysis[(dev_analysis["binary_label"] == 1) & (dev_analysis["pred"] == 0)]

print(f"=== Error Breakdown ===")
print(f"False Positives: {len(false_positives)} (model said PCL, actually not)")
print(f"False Negatives: {len(false_negatives)} (model missed PCL)")

print(f"\n--- Top 5 False Positives (highest confidence) ---")
for _, row in false_positives.nlargest(5, "prob").iterrows():
    print(f"  [{row['keyword']}] p={row['prob']:.3f} | label={row['label']}")
    print(f"  {row['text'][:150]}...")
    print()

print(f"--- Top 5 False Negatives (highest confidence wrong) ---")
for _, row in false_negatives.nsmallest(5, "prob").iterrows():
    print(f"  [{row['keyword']}] p={row['prob']:.3f} | label={row['label']}")
    print(f"  {row['text'][:150]}...")
    print()

In [ ]:
# Performance by community keyword
keyword_metrics = []
for kw in sorted(dev_analysis["keyword"].unique()):
    mask = dev_analysis["keyword"] == kw
    kw_labels = dev_analysis.loc[mask, "binary_label"].values
    kw_preds = dev_analysis.loc[mask, "pred"].values
    n_pos = kw_labels.sum()
    if n_pos > 0:
        kw_f1 = f1_score(kw_labels, kw_preds, pos_label=1, zero_division=0)
        kw_prec = precision_score(kw_labels, kw_preds, pos_label=1, zero_division=0)
        kw_rec = recall_score(kw_labels, kw_preds, pos_label=1, zero_division=0)
    else:
        kw_f1 = kw_prec = kw_rec = 0.0
    keyword_metrics.append({
        "keyword": kw, "n_total": mask.sum(), "n_positive": n_pos,
        "f1": kw_f1, "precision": kw_prec, "recall": kw_rec,
    })

kw_df = pd.DataFrame(keyword_metrics)
print("\n=== Performance by Community Keyword ===")
print(kw_df.to_string(index=False, float_format="{:.3f}".format))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(kw_df))
width = 0.25
ax.bar([i - width for i in x], kw_df["f1"], width, label="F1", color="steelblue")
ax.bar(x, kw_df["precision"], width, label="Precision", color="coral")
ax.bar([i + width for i in x], kw_df["recall"], width, label="Recall", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(kw_df["keyword"], rotation=45, ha="right")
ax.set_ylabel("Score")
ax.set_title("Performance by Community Keyword")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join("..", "report", "figures", "keyword_performance.png"), dpi=150)
plt.show()

In [ ]:
# Precision-Recall curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(dev_labels, dev_probs)
ap = average_precision_score(dev_labels, dev_probs)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall_curve, precision_curve, 'b-', linewidth=2, label=f'DeBERTa-v3 (AP={ap:.3f})')
ax.scatter(
    [recall_score(dev_labels, dev_preds, pos_label=1)],
    [precision_score(dev_labels, dev_preds, pos_label=1)],
    color='red', s=100, zorder=5, label=f'Operating point (t={best_threshold:.2f})'
)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join("..", "report", "figures", "pr_curve.png"), dpi=150)
plt.show()
print(f"Average Precision: {ap:.4f}")

## 10. Save Model

In [ ]:
# Save the best model and tokenizer
model_save_path = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Save threshold and config
import json
config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "community_aware": COMMUNITY_AWARE,
    "optimal_threshold": float(best_threshold),
    "dev_f1": float(best_f1),
    "focal_alpha": FOCAL_ALPHA,
    "focal_gamma": FOCAL_GAMMA,
    "seed": SEED,
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to {model_save_path}")
print(f"Config saved to {os.path.join(OUTPUT_DIR, 'config.json')}")
print(f"\nConfig: {json.dumps(config, indent=2)}")

## 11. Summary

### Approach
- **Model**: DeBERTa-v3-large with community-aware input
- **Loss**: Focal loss (alpha=0.75, gamma=2.0) + label smoothing (0.1)
- **Optimiser**: AdamW with layerwise LR decay (0.95)
- **Threshold**: Systematic F1-maximising threshold search
- **Ensemble**: Full ensemble (3 seeds, probability averaging) run via `src/train.py`

### Novel Components
1. **Community-aware input**: Prepended `[Community: keyword]` to encode community context
2. **Focal loss with label smoothing**: Handled 10:1 class imbalance without discarding data
3. **Layerwise LR decay**: Lower transformer layers preserve general knowledge
4. **Threshold optimisation**: Systematic F1-maximising threshold search
5. **Multi-seed ensemble**: Average probabilities from 3 independently trained models